# Experimental: Regime path (not production)

This notebook **consumes** production modules only. Do not redefine feature math or model contracts here.

Promote validated work into `features/`, `models/`, and `backtests/` with tests before API exposure.

In [ ]:
from features import build_default_store
from models.regime import ThresholdRegimeModel, synthetic_regime_panel
from backtests import run_walk_forward_classification
from strategies import regime_to_position, positions_to_returns

store = build_default_store()
raw, labels = synthetic_regime_panel(n=200, seed=42)
features = store.compute_many(["treasury_spread", "vix", "cpi_yoy"], raw)
features.tail()

In [ ]:
model = ThresholdRegimeModel(training_data_id="synthetic_regime_panel_v1")
model.fit(features, labels)
preds = model.predict(features)
preds.value_counts(), model.version.to_dict()

In [ ]:
asset_returns = raw["healthcare_etf"].pct_change().fillna(0.0)

def fit_predict(train_X, train_y, test_X):
    m = ThresholdRegimeModel()
    m.fit(train_X, train_y)
    return m.predict_codes(test_X)

def returns_from_preds(y_true, y_pred):
    label_map = {0: "risk_off", 1: "neutral", 2: "risk_on"}
    pos = regime_to_position(y_pred.map(label_map))
    return positions_to_returns(pos, asset_returns.loc[pos.index])

report = run_walk_forward_classification(
    features,
    labels,
    fit_predict,
    returns_from_preds=returns_from_preds,
    n_splits=4,
    model_name="threshold_regime",
    model_version="1.0.0",
)
report.to_dict()